# 07c · 오일러-마루야마와 밀스타인 (Euler–Maruyama and Milstein schemes: strong vs weak convergence)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Øksendal §5.1 (예제·해법), §5.2 (존재·유일성: Lipschitz·선형성장) · Higham (2001) SIAM Review 43(3) §4 (EM), §5 (strong/weak), §6 (Milstein) · Kloeden & Platen §9.6–9.7, §10.2, §10.3, §14.1 · Shreve II §6.2 (보조) |
| 선수 노트북 | 07a (Itô 적분·등거리성, $\int B\,dB$), 07b (Itô 공식, $dB^2=dt$), 06e (기하 브라운 운동) |
| 예상 소요 | 100분 |
| 상태 | draft |

07b 까지는 SDE 의 해를 **손으로** 구했다 (GBM, 지수 마팅게일). 대부분의 SDE 는 닫힌 해가 없으므로 수치 해법이 필요하다. 이 노트북은 가장 단순한 두 스킴 — 오일러-마루야마(Euler–Maruyama, EM)와 밀스타인(Milstein) — 을 쓰고, "수렴한다"는 말이 두 가지(강수렴 = 경로별, 약수렴 = 분포)로 갈라진다는 것을 Higham (2001) 의 실험을 재현하며 몸으로 익힌다. GBM 에서는 두 스킴의 평균제곱 강오차가 **정확한 닫힌 형식**으로 나오므로 (정리 1, 완전 증명) 시뮬레이션을 4 SE 안에서 검증할 수 있다. 마지막으로 Lipschitz 가정을 깨서 EM 이 폭발하는 것을 본다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.special import gamma
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import MCResult, mc_estimate, mc_proportion, compare_table, assert_close
from spkit.brownian import brownian_paths, coarsen
from spkit.sde import euler_maruyama, milstein
from spkit.plots import plot_paths

SEED, rng = rng_for("07c")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기 (강수렴 실험)
N_WEAK = scale(50_000)    # 약수렴 실험
N_FINE = 1024             # 세밀 격자 스텝 수 (Δt_fine = 2^-10)
T = 1.0
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_WEAK={N_WEAK}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (07b) Itô 공식으로 $d(B_t^2)$ 과 $d(e^{B_t})$ 를 써라.

<details><summary>정답</summary>

$d(B^2)=2B\,dB+dt$; $d(e^B)=e^B\,dB+\frac12e^B\,dt$. 둘 다 $\frac12f''\,dt$ 항이 $dB^2=dt$ 에서 나온다 — 오늘 밀스타인 보정항의 출처가 바로 이것이다.

</details>

**Q2.** (07b) GBM $dS=\mu S\,dt+\sigma S\,dB$ 의 해와 $E[\log S_T]$ 는?

<details><summary>정답</summary>

$S_T=S_0\exp\big((\mu-\sigma^2/2)T+\sigma B_T\big)$, $E[\log S_T]=\log S_0+(\mu-\sigma^2/2)T$. 오늘은 이 정확해를 **기준 답**으로 두고 수치 스킴의 오차를 잰다 — $S_T$ 가 $B_T$ 하나로 결정되므로 격자 해상도와 무관하게 한 번만 계산하면 된다.

</details>

**Q3.** (07b) 독립 BM 두 개의 이산 교차변동 $\sum_i\Delta B_i\Delta W_i$ 의 평균제곱은?

<details><summary>정답</summary>

$T^2/n\to0$ (교차변동 0, 곱셈 규칙에 $dB\,dW$ 항 없음). 오늘의 함정과 연결: 강오차를 잴 때 두 근사가 **독립** 경로를 쓰면 교차항이 사라져 오차가 $O(1)$ 로 남는다.

</details>

**Q4.** (07a) Itô 등거리성을 써서 $E\big[(\sum_i\xi_i\Delta B_i)^2\big]$ 을 $\xi_i$ 로 표현하라.

<details><summary>정답</summary>

$\sum_iE[\xi_i^2]\,\Delta t$ ($\xi_i$ 가 $\mathcal F_{t_i}$-적합일 때; 교차항은 독립 증분의 평균 0 으로 사라진다). EM 의 국소오차를 스텝 수 $n$ 개 더할 때 같은 논리로 제곱이 **합**으로 쌓인다 — 그래서 전역 RMS 오차가 $\sqrt{n\cdot\Delta t^2}=\sqrt{T\Delta t}$, 즉 차수 $1/2$ 가 된다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **SDE 와 강해(strong solution).** $dX_t=a(t,X_t)\,dt+b(t,X_t)\,dB_t$, $X_0=x_0$. 주어진 BM 경로 $B$ 에 적합한(adapted) 해 $X$ 를 강해라 한다. 존재·유일성: $a,b$ 가 $x$ 에 대해 전역 Lipschitz 이고 선형성장 조건 $|a|+|b|\le C(1+|x|)$ 을 만족하면 (Øksendal §5.2 Thm 5.2.1).
- **오일러-마루야마(EM).** 격자 $t_k=k\Delta t$, $\Delta B_k=B_{t_{k+1}}-B_{t_k}\sim N(0,\Delta t)$ 독립.
  $$X_{k+1}=X_k+a(t_k,X_k)\Delta t+b(t_k,X_k)\Delta B_k.$$
- **밀스타인(Milstein, 스칼라).** $b_x=\partial b/\partial x$ 로
  $$X_{k+1}=X_k+a\,\Delta t+b\,\Delta B_k+\tfrac12\,b\,b_x\,(\Delta B_k^2-\Delta t).$$
  가법 잡음($b_x\equiv0$)이면 EM 과 같다.
- **강수렴 차수(strong order) $\gamma$.** $E|X_T-X_T^{\Delta t}|\le C\Delta t^{\gamma}$ — **같은 BM 경로**로 비교하는 경로별 오차. 이 노트북은 평균제곱 버전 $\sqrt{E[(X_T-X_T^{\Delta t})^2]}\le C\Delta t^\gamma$ 도 함께 쓴다 (Jensen: $E|Y|\le\sqrt{E[Y^2]}$ 이므로 $L^1$ 차수 $\ge$ $L^2$ 차수).
- **약수렴 차수(weak order) $\beta$.** $|E f(X_T)-E f(X_T^{\Delta t})|\le C\Delta t^\beta$ (충분히 매끄럽고 다항 성장인 $f$). 분포만 맞으면 되므로 경로를 공유할 필요가 없다.
- **같은 경로의 coarsen.** 세밀 격자 $\Delta t_{\text{fine}}=T/N$ 의 경로 $B$ 에서 `coarsen(t, B, factor)` 는 매 factor 번째 점을 취한 **같은 경로**의 저해상도 버전이고, 그 증분 `np.diff` 는 세밀 증분 factor 개의 합이다. 마지막 점 $B_T$ 는 모든 해상도에서 동일하므로 정확해 $X_T=X_0\exp((\mu-\sigma^2/2)T+\sigma B_T)$ 는 한 번만 계산한다.

### 2.2 정리 1 — GBM 에 대한 EM·밀스타인의 평균제곱 강오차 (정확한 닫힌 형식)

$dX=\mu X\,dt+\sigma X\,dB$, $X_0=1$, $n=T/\Delta t$. 같은 증분 $\Delta B_k$ 로 계산한 EM 근사 $X_n^{EM}$ 과 정확해 $X_T$ 에 대해

$$E\big[(X_n^{EM}-X_T)^2\big]=\big(1+(2\mu+\sigma^2)\Delta t+\mu^2\Delta t^2\big)^n-2e^{\mu T}\big(1+(\mu+\sigma^2)\Delta t\big)^n+e^{(2\mu+\sigma^2)T}.$$

밀스타인 ($a:=1+\mu\Delta t-\frac12\sigma^2\Delta t$):

$$E\big[(X_n^{Mil}-X_T)^2\big]=\big(a^2+(1+a)\sigma^2\Delta t+\tfrac34\sigma^4\Delta t^2\big)^n-2e^{\mu T}\big(1+(\mu+\sigma^2)\Delta t+\tfrac12\sigma^4\Delta t^2\big)^n+e^{(2\mu+\sigma^2)T}.$$

따라서 $\Delta t\to0$ 에서 EM 의 MSE 는 $\frac{\sigma^4}{2}Te^{(2\mu+\sigma^2)T}\Delta t+O(\Delta t^2)$ (RMS $\asymp\Delta t^{1/2}$), 밀스타인의 MSE 는 $O(\Delta t^2)$ (RMS $\asymp\Delta t$).

가정이 왜 필요한가:
- **증분 $\Delta B_k$ 가 독립·$N(0,\Delta t)$** (BM 정의): 곱의 기댓값이 인수별 기댓값의 곱으로 분해되는 유일한 이유.
- **선형 SDE** (계수가 $x$ 에 비례): 한 스텝의 인수 $F_k$ 가 $X_k$ 와 독립이 되어 $X_n=\prod F_k$, $X_T=\prod G_k$ 로 쓸 수 있다. 비선형이면 닫힌 형식이 없다 — 그래서 일반 정리(정리 2, Kloeden–Platen Thm 10.2.2)는 Gronwall 부등식으로 **상한**만 준다.
- **같은 경로**: $E[X_nX_T]$ 계산에서 $F_k$ 와 $G_k$ 가 같은 $\Delta B_k$ 를 공유해야 한다. 독립 경로를 쓰면 교차항이 $E[X_n]E[X_T]=e^{\mu T}(1+\mu\Delta t)^n$ 으로 바뀌어 오차가 $\Delta t\to0$ 에서도 $2e^{2\mu T}(e^{\sigma^2T}-1)$ 로 남는다 (강오차 측정 불가능).

<details><summary>증명</summary>

**0. 곱 구조.** EM 한 스텝은 $X_{k+1}=X_k(1+\mu\Delta t+\sigma\Delta B_k)=:X_kF_k$ 이므로 $X_n^{EM}=\prod_{k<n}F_k$. 정확해는 $X_T=\exp\big((\mu-\frac{\sigma^2}2)T+\sigma B_T\big)=\prod_{k<n}G_k$, $G_k=\exp\big((\mu-\frac{\sigma^2}{2})\Delta t+\sigma\Delta B_k\big)$. $F_k,G_k$ 는 $\Delta B_k$ 만의 함수이고 $\Delta B_k$ 들이 독립이므로 (곱의 기댓값 = 기댓값의 곱)

$$E[(X_n-X_T)^2]=E[X_n^2]-2E[X_nX_T]+E[X_T^2]=\big(E[F^2]\big)^n-2\big(E[FG]\big)^n+\big(E[G^2]\big)^n,$$

여기서 $F,G$ 는 한 스텝 인수, $\Delta B\sim N(0,\Delta t)$. 세 기댓값을 계산하면 된다.

**(i) $E[F^2]$.** $F=1+\mu\Delta t+\sigma\Delta B$, $E[\Delta B]=0$, $E[\Delta B^2]=\Delta t$ 이므로 $E[F^2]=(1+\mu\Delta t)^2+\sigma^2\Delta t=1+(2\mu+\sigma^2)\Delta t+\mu^2\Delta t^2$.

**(ii) $E[G^2]$.** $E[e^{2\sigma\Delta B}]=e^{2\sigma^2\Delta t}$ (정규 mgf) 이므로 $E[G^2]=e^{(2\mu-\sigma^2)\Delta t}e^{2\sigma^2\Delta t}=e^{(2\mu+\sigma^2)\Delta t}$, 따라서 $(E[G^2])^n=e^{(2\mu+\sigma^2)T}$.

**(iii) $E[FG]$ — 기울인 측도.** $M:=e^{\sigma\Delta B-\sigma^2\Delta t/2}$ 라 두면 $E[M]=1$ (07b E1 의 지수 마팅게일) 이고 $G=e^{\mu\Delta t}M$. 밀도를 직접 보면
$$e^{\sigma x-\sigma^2\Delta t/2}\cdot\frac{e^{-x^2/(2\Delta t)}}{\sqrt{2\pi\Delta t}}=\frac{e^{-(x-\sigma\Delta t)^2/(2\Delta t)}}{\sqrt{2\pi\Delta t}},$$
즉 $M$ 을 곱해 기댓값을 취하는 것은 $\Delta B$ 를 $N(\sigma\Delta t,\Delta t)$ 로 바꿔 기댓값을 취하는 것과 같다 (이것이 측도변환 / Girsanov 의 한 스텝 버전). 따라서
$$E[\Delta B\,M]=\sigma\Delta t,\qquad E[\Delta B^2M]=\mathrm{Var}+\text{평균}^2=\Delta t+\sigma^2\Delta t^2 .$$
그러므로 $E[FG]=e^{\mu\Delta t}\big(1+\mu\Delta t+\sigma\cdot\sigma\Delta t\big)=e^{\mu\Delta t}\big(1+(\mu+\sigma^2)\Delta t\big)$, 그리고 $(E[FG])^n=e^{\mu T}(1+(\mu+\sigma^2)\Delta t)^n$. 세 항을 모으면 EM 식이다.

**(iv) 밀스타인.** $b=\sigma x$, $b_x=\sigma$ 이므로 한 스텝 인수는 $F=1+\mu\Delta t+\sigma\Delta B+\frac12\sigma^2(\Delta B^2-\Delta t)=a+\sigma\Delta B+\frac12\sigma^2\Delta B^2$, $a=1+\mu\Delta t-\frac12\sigma^2\Delta t$. 정규 모멘트 $E\Delta B=E\Delta B^3=0$, $E\Delta B^2=\Delta t$, $E\Delta B^4=3\Delta t^2$ 로
$$E[F^2]=a^2+\sigma^2\Delta t+\tfrac14\sigma^4\cdot3\Delta t^2+2a\cdot\tfrac12\sigma^2\Delta t=a^2+(1+a)\sigma^2\Delta t+\tfrac34\sigma^4\Delta t^2 .$$
(iii) 의 기울인 측도로 $E[FG]=e^{\mu\Delta t}\big(a+\sigma\cdot\sigma\Delta t+\frac12\sigma^2(\Delta t+\sigma^2\Delta t^2)\big)=e^{\mu\Delta t}\big(1+(\mu+\sigma^2)\Delta t+\frac12\sigma^4\Delta t^2\big)$. $E[G^2]$ 은 그대로. 모으면 밀스타인 식이다.

**(v) 차수.** 각 항의 로그를 $\Delta t$ 의 2차까지 전개한다 ($n\Delta t=T$). EM:
$n\log E[F^2]=(2\mu+\sigma^2)T+\big(\mu^2-\tfrac12(2\mu+\sigma^2)^2\big)T\Delta t+O(\Delta t^2)$,
$\log\big(e^{\mu T}(E[FG]/e^{\mu\Delta t})^n\big)=(2\mu+\sigma^2)T-\tfrac12(\mu+\sigma^2)^2T\Delta t+O(\Delta t^2)$.
두 계수의 차이 $c_1-2c_2$ 를 계산하면 $\big(-\mu^2-2\mu\sigma^2-\frac{\sigma^4}{2}\big)-2\big(-\frac{\mu^2}{2}-\mu\sigma^2-\frac{\sigma^4}{2}\big)=\frac{\sigma^4}{2}$ 이므로
$$\text{MSE}_{EM}=e^{(2\mu+\sigma^2)T}\big[e^{c_1T\Delta t}-2e^{c_2T\Delta t}+1\big]=e^{(2\mu+\sigma^2)T}\,\tfrac{\sigma^4}{2}\,T\Delta t+O(\Delta t^2).$$
밀스타인은 같은 계산에서 $E[F^2]=1+(2\mu+\sigma^2)\Delta t+(\mu^2+\frac12\sigma^4)\Delta t^2$ 이라 $c_1=-(\mu^2+2\mu\sigma^2)$, $c_2=-(\frac12\mu^2+\mu\sigma^2)$, 즉 $c_1-2c_2=0$: $O(\Delta t)$ 항이 **소거**되어 MSE $=O(\Delta t^2)$. $\blacksquare$

</details>

### 2.3 정리 2 — EM 의 강수렴 차수 1/2 (일반 정리, cite)

$a,b$ 가 전역 Lipschitz·선형성장이고 $E[X_0^2]<\infty$ 이면, EM 의 연속 보간 $\bar X^{\Delta t}$ 에 대해
$$\Big(E\big[\sup_{t\le T}|X_t-\bar X_t^{\Delta t}|^2\big]\Big)^{1/2}\le C\,\Delta t^{1/2}.$$
가법 잡음($b_x=0$)이면 차수 1. 밀스타인은 ($b$ 가 $C^2$ 등 추가 매끄러움 아래) 차수 1.

가정이 왜 필요한가:
- **Lipschitz**: 오차 재귀식에서 $|a(x)-a(y)|\le K|x-y|$ 로 Gronwall 부등식을 적용하기 위해. 초선형 성장($-x^3$)이면 EM 이 유한 확률로 폭발한다 (Step 4, E3; Hutzenthaler–Jentzen–Kloeden 2011).
- **선형성장**: 해의 모멘트 유계 $E\sup_t|X_t|^2<\infty$ 를 보장 — 오차의 제곱평균이 정의되려면 필요.
- **차수 $1/2$ 의 출처**: 한 스텝에서 $b(X_s)-b(X_{t_k})\approx b_xb\,(B_s-B_{t_k})$ 를 무시한 국소오차의 제곱평균이 $O(\Delta t^2)$ 이고, 스텝 수가 $n=T/\Delta t$ 이므로 (Q4 처럼 제곱이 합으로 쌓여) 전역 MSE $O(\Delta t)$ → RMS $O(\Delta t^{1/2})$. 밀스타인은 바로 이 항을 되돌려 넣어 국소오차를 $O(\Delta t^3)$ 으로 낮춘다.

증명: Kloeden & Platen Thm 10.2.2 (EM), Thm 10.3.5 (밀스타인); 간결한 증명은 Mao, *Stochastic Differential Equations and Applications*, Thm 2.7.3.

**밀스타인 보정항의 유도 (sketch).** $[t_k,t_{k+1}]$ 에서 $\int_{t_k}^{t_{k+1}}b(X_s)\,dB_s$ 를 근사할 때 EM 은 $b(X_s)\approx b(X_{t_k})$ 로 얼린다. 한 단계 더: $b(X_s)$ 에 Itô 공식(07b)을 적용하면 $db(X_s)=(\dots)ds+b_x(X_s)\,b(X_s)\,dB_s$ 이고, $ds$ 항은 $O(\Delta t)$ 로 작으므로 $b(X_s)\approx b(X_{t_k})+b_x(X_{t_k})\,b(X_{t_k})\,(B_s-B_{t_k})$. 이것을 넣으면
$$\int_{t_k}^{t_{k+1}}b(X_s)dB_s\approx b\,\Delta B_k+b\,b_x\int_{t_k}^{t_{k+1}}(B_s-B_{t_k})\,dB_s=b\,\Delta B_k+\tfrac12\,b\,b_x\,(\Delta B_k^2-\Delta t),$$
마지막 등호가 07a 의 $\int_0^tB\,dB=\frac12(B_t^2-t)$ 를 구간 $[t_k,t_{k+1}]$ 에 적용한 것이다. 즉 밀스타인 보정항 $=$ "$dB^2=dt$".

### 2.4 정리 3 — EM 의 약수렴 차수 1 (GBM 의 평균에 대한 정확 계산)

$dX=\mu X\,dt+\sigma X\,dB$, $X_0=x_0$ 에서
$$E[X_n^{EM}]=x_0(1+\mu\Delta t)^n\ (\sigma\text{ 에 무관}),\qquad E[(X_n^{EM})^2]=x_0^2\big(1+(2\mu+\sigma^2)\Delta t+\mu^2\Delta t^2\big)^n .$$
따라서 $\big|E[X_T]-E[X_n^{EM}]\big|=x_0\big|e^{\mu T}-(1+\mu\Delta t)^n\big|=x_0e^{\mu T}\frac{\mu^2T}{2}\Delta t+O(\Delta t^2)$: **약차수 1**. 밀스타인의 평균도 $x_0(1+\mu\Delta t)^n$ 으로 동일 (보정항의 평균이 0) → 밀스타인은 약차수를 높이지 않는다.

가정이 왜 필요한가:
- $E[\Delta B_k]=0$, $E[\Delta B_k^2-\Delta t]=0$ 과 $X_k$ 와의 **독립성**: 평균이 곱으로 분해되는 이유. (EM 이 $\Delta B_k$ 를 $X_k$ 이후에 뽑는다는 뜻 — 적합성.)
- $f(x)=x$ 는 매끄럽고 다항 성장: 일반 약수렴 정리 (Kloeden–Platen §14.1) 의 가정. 일반 $f$ 에서는 정확 계산 대신 Itô–Taylor 전개의 약한 잔차로 같은 차수 1 을 얻는다.

<details><summary>증명</summary>

**평균.** $E[X_{k+1}]=E[X_k](1+\mu\Delta t)+\sigma E[X_k\Delta B_k]=E[X_k](1+\mu\Delta t)$ — $X_k$ 는 $\Delta B_0,\dots,\Delta B_{k-1}$ 의 함수라 $\Delta B_k$ 와 독립이고 $E[\Delta B_k]=0$. 귀납으로 $E[X_n]=x_0(1+\mu\Delta t)^n$.

**약오차의 차수.** $n\log(1+\mu\Delta t)=n\big(\mu\Delta t-\frac{\mu^2\Delta t^2}{2}+O(\Delta t^3)\big)=\mu T-\frac{\mu^2T}{2}\Delta t+O(\Delta t^2)$, 따라서 $(1+\mu\Delta t)^n=e^{\mu T}\big(1-\frac{\mu^2T}{2}\Delta t+O(\Delta t^2)\big)$ 이고 $e^{\mu T}-(1+\mu\Delta t)^n=e^{\mu T}\frac{\mu^2T}{2}\Delta t+O(\Delta t^2)$.

**제곱.** $E[X_{k+1}^2]=E[X_k^2]\,E[(1+\mu\Delta t+\sigma\Delta B_k)^2]=E[X_k^2]\big(1+(2\mu+\sigma^2)\Delta t+\mu^2\Delta t^2\big)$ (같은 독립성 + (i)). 귀납.

**밀스타인.** 한 스텝 인수의 평균 $E\big[1+\mu\Delta t+\sigma\Delta B+\frac12\sigma^2(\Delta B^2-\Delta t)\big]=1+\mu\Delta t$ — 보정항은 평균 0. 따라서 $E[X_n^{Mil}]=x_0(1+\mu\Delta t)^n=E[X_n^{EM}]$. $\blacksquare$

</details>

### 2.5 직관

EM 은 "한 스텝 동안 계수를 얼려 놓고" 드리프트는 $\Delta t$, 잡음은 $\Delta B\sim\sqrt{\Delta t}$ 만큼 움직인다. 잡음 계수 $b$ 가 스텝 안에서 변하는 것을 무시한 대가가 스텝당 $\sim bb_x(\Delta B)^2\sim\Delta t$ 이고, 이것은 부호가 랜덤이 아니라 $\frac12bb_x(\Delta B^2-\Delta t)$ 라는 구조를 가진다 — 07b 의 $dB^2=dt$ 그 자체다. 밀스타인은 이 항을 되돌려 넣는다.

반면 **분포**(약수렴)만 보면 $\Delta B^2-\Delta t$ 의 평균이 0 이라 이 보정은 아무 도움이 안 되고, EM 의 평균은 그냥 결정론적 오일러 $(1+\mu\Delta t)^n$ 이다. 옵션 가격(기댓값)을 계산할 때와 헤지 경로(경로별)를 추적할 때 필요한 "정확도"가 다르다는 뜻이다.

강오차를 재려면 두 근사가 **같은** 브라운 경로를 봐야 한다: 세밀 경로를 만들고 `coarsen` 으로 증분을 합쳐 쓰는 것이 Higham (2001) 실험의 핵심 트릭이다.

### 2.6 함정 (traps)

- 강오차를 잴 때 각 $\Delta t$ 마다 **새 난수**를 뽑으면 오차가 $O(1)$ 로 남는다 — 반드시 같은 경로를 coarsen 해서 써야 한다.
- log-log 기울기는 **점근값**이다: 유한 $\Delta t\in[2^{-10},2^{-5}]$ 범위에서는 정확 MSE 곡선의 최소제곱 기울기조차 0.5 가 아니라 약 0.57 (EM) 이다. 기울기 판정은 범위($\pm0.15$)로 한다.
- 밀스타인은 **강차수만** 1 로 올린다. 기댓값(옵션 가격 등)만 필요하면 EM 으로 충분하고, 오히려 경로 수(MC 오차 $N^{-1/2}$)가 병목이다.
- Higham 원본의 $\mu=2,\sigma=1$ 은 $X_T$ 의 4차 모멘트가 $e^{(4\mu+6\sigma^2)T}=e^{14}$ 라 강오차 추정의 상대 SE 가 10–30% 로 커서 4 SE 판정이 무의미해진다 — 그래서 여기서는 $\mu=1,\sigma=0.5$ 를 쓴다 (E1c). 파라미터 선택도 실험 설계다.
- 약오차 실험에서 MC 잡음(SE ≈ 0.003 at 50 000 경로) 이 가장 작은 약오차(0.057) 의 5% 여야 하므로 $\sigma=0.1$ 로 분산을 줄인다; $\sigma=0.5$ 로 두면 잡음이 약오차를 덮는다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — 특히 '강수렴과 약수렴의 차이'를 옵션 가격 계산(기댓값)과 헤지 경로 추적(경로별)의 예로 각각 1문장씩, 그리고 왜 강오차 측정에는 같은 브라운 경로가 필요한지 1문장으로.)

## 3. Predict — 코드를 돌리기 전에 예측

정리 1·3 의 식에 숫자를 넣으면 **손으로**(또는 계산기로) 구할 수 있습니다. 시뮬레이션 뒤 5절의 표에서 자기 예측과 비교합니다.

In [ ]:
predictions = {
    # dX = X dt + 0.5 X dB, X_0 = 1, T = 1: E[X_1] 은?
    "exact_mean": None,
    # 같은 SDE 를 EM, Δt = 1/16 으로 풀면 E[X_16^EM] 은? (정리 3)
    "em_mean_dt16": None,
    # EM, Δt = 1/32 의 RMS 강오차 sqrt(E[(X^EM_T - X_T)^2]) 은? (정리 1 의 닫힌 형식)
    "rmse_em_dt32": None,
    # 밀스타인, Δt = 1/32 의 RMS 강오차는?
    "rmse_mil_dt32": None,
    # Δt ∈ {2^-10, …, 2^-5} 에서 log RMS vs log Δt 의 최소제곱 기울기 — EM 은 대략?
    "slope_em": None,
    # 같은 기울기 — 밀스타인은 대략?
    "slope_mil": None,
    # 약오차 실험 (μ=2, σ=0.1): Δt = 2^-8 에서 |E[X_1] - E[X^EM_256]| 은?
    "weak_err_dt256": None,
}

## 4. Simulate — 시뮬레이션

먼저 정리 1 의 닫힌 형식과 작은 도우미를 코드로 옮긴다. 이 노트북에서 "정확"이라 부르는 숫자는 모두 여기서 계산된다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def mse_em_exact(mu, sigma, dt, T=1.0, x0=1.0):
    """정리 1: GBM 에 대한 EM 의 평균제곱 강오차 E[(X_n^EM - X_T)^2] (같은 경로)."""
    n = T / dt        # 스텝 수 (dt 가 T 를 나누면 정수; 아니면 매끄러운 보간 — 그림의 정확 곡선용)
    return x0**2 * ((1 + (2 * mu + sigma**2) * dt + mu**2 * dt**2) ** n
                    - 2 * np.exp(mu * T) * (1 + (mu + sigma**2) * dt) ** n
                    + np.exp((2 * mu + sigma**2) * T))

def mse_mil_exact(mu, sigma, dt, T=1.0, x0=1.0):
    """정리 1: GBM 에 대한 밀스타인의 평균제곱 강오차."""
    n = T / dt
    a = 1 + mu * dt - 0.5 * sigma**2 * dt
    return x0**2 * ((a**2 + (1 + a) * sigma**2 * dt + 0.75 * sigma**4 * dt**2) ** n
                    - 2 * np.exp(mu * T) * (1 + (mu + sigma**2) * dt + 0.5 * sigma**4 * dt**2) ** n
                    + np.exp((2 * mu + sigma**2) * T))

def loglog_slope(x, y):
    """log y vs log x 의 최소제곱 기울기."""
    return np.polyfit(np.log(x), np.log(y), 1)[0]

def explosion_fraction(X):
    """non-finite 이거나 |X| > 1e3 인 성분의 비율 (Step 4 / E3)."""
    return np.mean(~np.isfinite(X) | (np.abs(X) > 1e3))

### Step 1 — 그림으로 보는 EM: 같은 경로, 세 해상도

강오차 파라미터 $\mu=1,\sigma=0.5,X_0=1,T=1$. 세밀 격자($\Delta t=2^{-10}$)의 브라운 경로 하나를 만들고, 정확해 $X_t=\exp((\mu-\sigma^2/2)t+\sigma B_t)$ 를 전 격자에서 계산한 뒤, **같은 경로**를 factor $\in\{64,16,1\}$ 로 coarsen 한 증분을 `dW` 로 넘겨 EM 을 돌린다 (`dW` 를 넘기면 `rng` 는 소비되지 않는다).

In [ ]:
mu, sigma, X0 = 1.0, 0.5, 1.0
a = lambda t, x: mu * x            # 드리프트 a(t, x)
b = lambda t, x: sigma * x         # 확산 b(t, x)
bx = lambda t, x: sigma + 0 * x    # ∂b/∂x (밀스타인용)

t, B = brownian_paths(1, N_FINE, T, rng)
X_exact = np.exp((mu - 0.5 * sigma**2) * t + sigma * B)   # 정확해, 전 격자

fig, ax = plt.subplots()
ax.plot(t, X_exact[0], "k", lw=2, label="exact GBM on the fine path")
for factor, style in [(64, "o-"), (16, "s-"), (1, "-")]:
    tk, Bk = coarsen(t, B, factor)
    dW = np.diff(Bk, axis=1)
    tk2, Xk = euler_maruyama(a, b, X0, T, N_FINE // factor, rng, dW=dW)
    ax.plot(tk2, Xk[0], style, ms=4, lw=1, alpha=0.8, label=f"EM, Δt = 1/{N_FINE // factor}")
    print(f"Δt = 1/{N_FINE // factor:5d}: X_T^EM = {Xk[0, -1]:.4f}   (exact X_T = {X_exact[0, -1]:.4f})")
ax.set(xlabel="t", ylabel="X_t", title="Euler–Maruyama at Δt = 1/16, 1/64, 1/1024 on the SAME Brownian path vs exact GBM")
ax.legend(loc="upper left")
plt.show()

굵은 격자의 EM 은 정확해의 큰 굴곡은 따라가지만 끝값이 어긋난다; 격자가 세밀해질수록 검은 곡선에 붙는다. 이 "붙는 속도"가 강수렴 차수다.

### Step 2 — 강수렴: Higham `em_strong` / `milstrong` 재현

절차: (1) 세밀 경로 `B` 를 `N_PATHS` 개 생성 — **이것이 이 실험의 유일한 난수 소비**. (2) 정확해 $X_T$ 는 $B_T$ 로 한 번만 계산. (3) factor $=2^k$ ($k=0..5$, $\Delta t=2^{-10}..2^{-5}$) 마다 coarsen 한 증분으로 EM·밀스타인을 돌리고 오차의 제곱·절댓값을 `mc_estimate` 로 기록. 각 수준에서 `Bk[:, -1] == B[:, -1]` 을 assert 해 같은 경로임을 명시적으로 확인한다.

In [ ]:
t, B = brownian_paths(N_PATHS, N_FINE, T, rng)
X_T = np.exp((mu - 0.5 * sigma**2) * T + sigma * B[:, -1])   # 정확해 (모든 해상도 공통)

factors = [1, 2, 4, 8, 16, 32]
dts = np.array([f / N_FINE for f in factors])                 # 2^-10 … 2^-5
mse_em, mse_mil, l1_em, l1_mil = [], [], [], []
for factor in factors:
    tk, Bk = coarsen(t, B, factor)
    assert np.allclose(Bk[:, -1], B[:, -1])                   # 같은 경로 (B_T 동일)
    dW = np.diff(Bk, axis=1)                                  # 굵은 증분 = 세밀 증분 factor 개의 합
    _, Xe = euler_maruyama(a, b, X0, T, N_FINE // factor, rng, dW=dW)
    _, Xm = milstein(a, b, bx, X0, T, N_FINE // factor, rng, dW=dW)
    err_e, err_m = Xe[:, -1] - X_T, Xm[:, -1] - X_T
    mse_em.append(mc_estimate(err_e**2));  mse_mil.append(mc_estimate(err_m**2))
    l1_em.append(mc_estimate(np.abs(err_e)));  l1_mil.append(mc_estimate(np.abs(err_m)))

print(f"{'Δt':>7} | {'EM MSE (MC)':>12} {'exact':>10} {'z':>5} | {'Mil MSE (MC)':>13} {'exact':>10} {'z':>5}")
for i, dt in enumerate(dts):
    ex_e, ex_m = mse_em_exact(mu, sigma, dt), mse_mil_exact(mu, sigma, dt)
    ze, zm = (mse_em[i].mean - ex_e) / mse_em[i].se, (mse_mil[i].mean - ex_m) / mse_mil[i].se
    print(f"2^-{10 - i:<3} | {mse_em[i].mean:12.6f} {ex_e:10.6f} {ze:5.2f} | {mse_mil[i].mean:13.7f} {ex_m:10.7f} {zm:5.2f}")

여섯 수준 모두 $|z|$ 가 2 안팎 이하면 정리 1 의 닫힌 형식이 시뮬레이션과 맞는 것이다 (공식 검증은 5절). 이제 RMS 를 log-log 로 그려 기울기를 읽는다. MC RMS 의 오차막대는 델타법 $\mathrm{SE}(\sqrt{m})\approx\mathrm{SE}(m)/(2\sqrt m)$ 으로 그린다.

In [ ]:
rms_em = np.sqrt([m.mean for m in mse_em]);  rms_mil = np.sqrt([m.mean for m in mse_mil])
se_rms_em = np.array([m.se for m in mse_em]) / (2 * rms_em)
se_rms_mil = np.array([m.se for m in mse_mil]) / (2 * rms_mil)
dt_fine = np.logspace(-10, -5, 60, base=2)
exact_rms_em = np.sqrt([mse_em_exact(mu, sigma, d) for d in dt_fine])
exact_rms_mil = np.sqrt([mse_mil_exact(mu, sigma, d) for d in dt_fine])
slope_em_rms, slope_mil_rms = loglog_slope(dts, rms_em), loglog_slope(dts, rms_mil)
slope_em_exact = loglog_slope(dts, np.sqrt([mse_em_exact(mu, sigma, d) for d in dts]))
slope_mil_exact = loglog_slope(dts, np.sqrt([mse_mil_exact(mu, sigma, d) for d in dts]))

fig, ax = plt.subplots()
ax.errorbar(dts, rms_em, yerr=2 * se_rms_em, fmt="o", capsize=3, label=f"EM RMS (MC ± 2 SE), slope {slope_em_rms:.2f}")
ax.errorbar(dts, rms_mil, yerr=2 * se_rms_mil, fmt="s", capsize=3, label=f"Milstein RMS (MC ± 2 SE), slope {slope_mil_rms:.2f}")
ax.plot(dt_fine, exact_rms_em, "-", color="C0", lw=1, alpha=0.6, label="exact closed form (EM)")
ax.plot(dt_fine, exact_rms_mil, "-", color="C1", lw=1, alpha=0.6, label="exact closed form (Milstein)")
ax.plot(dts, rms_em[-1] * (dts / dts[-1]) ** 0.5, ":", color="k", label="reference slope 1/2")
ax.plot(dts, rms_mil[-1] * (dts / dts[-1]) ** 1.0, "--", color="k", label="reference slope 1")
ax.set(xscale="log", yscale="log", xlabel="Δt", ylabel="RMS strong error at T=1",
       title="Strong error vs Δt (log-log): EM and Milstein RMS error with exact curves")
ax.legend(fontsize=8)
plt.show()
print(f"MC slopes: EM {slope_em_rms:.3f}, Milstein {slope_mil_rms:.3f}   |   exact-curve slopes on the same range: {slope_em_exact:.3f}, {slope_mil_exact:.3f}")

EM 점들은 기울기 $1/2$ 참조선과 나란하고, 밀스타인은 기울기 1 참조선과 나란하다. 주의: 정확 곡선 자체의 이 범위 기울기가 0.57 이다 — $O(\Delta t^2)$ 항이 $\Delta t=2^{-5}$ 에서 아직 보이기 때문이며, MC 기울기가 0.5 와 정확히 같지 않은 것은 잡음이 아니라 유한-$\Delta t$ 효과다. Higham 이 쓴 $L^1$ 오차 $E|X_T^{\Delta t}-X_T|$ 도 같은 차수를 준다 (Jensen 으로 $L^1\le L^2$):

In [ ]:
l1e = np.array([m.mean for m in l1_em]);  l1m = np.array([m.mean for m in l1_mil])
slope_em_l1, slope_mil_l1 = loglog_slope(dts, l1e), loglog_slope(dts, l1m)
fig, ax = plt.subplots()
ax.errorbar(dts, l1e, yerr=2 * np.array([m.se for m in l1_em]), fmt="o", capsize=3, label=f"EM  E|X^Δ_T − X_T|, slope {slope_em_l1:.2f}")
ax.errorbar(dts, l1m, yerr=2 * np.array([m.se for m in l1_mil]), fmt="s", capsize=3, label=f"Milstein  E|X^Δ_T − X_T|, slope {slope_mil_l1:.2f}")
ax.plot(dts, l1e[-1] * (dts / dts[-1]) ** 0.5, ":", color="k", label="reference slope 1/2")
ax.plot(dts, l1m[-1] * (dts / dts[-1]) ** 1.0, "--", color="k", label="reference slope 1")
ax.set(xscale="log", yscale="log", xlabel="Δt", ylabel="L1 strong error at T=1",
       title="Strong L1 error E|X^Δ_T − X_T| vs Δt (Higham's measure), MC ± 2 SE")
ax.legend(fontsize=8)
plt.show()

### Step 3 — 약수렴: Higham `em_weak` 재현

파라미터 $\mu=2,\sigma=0.1,X_0=1,T=1$ (분산이 작아 약오차가 MC 잡음 위로 보인다). 약수렴은 **분포**의 오차이므로 경로를 공유할 필요가 없다 — 각 $\Delta t$ 마다 새 난수를 뽑는다. 정리 3 으로 $E[X_n^{EM}]=(1+\mu\Delta t)^n$ 이 정확하므로, MC 평균이 이 값과 맞는지(구현 검증)와 $|e^{\mu T}-(1+\mu\Delta t)^n|$ 이 $\Delta t$ 에 비례하는지(약차수 1) 를 따로 본다.

In [ ]:
mu2, sigma2 = 2.0, 0.1
a2 = lambda t, x: mu2 * x
b2 = lambda t, x: sigma2 * x
bx2 = lambda t, x: sigma2 + 0 * x

weak_ps = list(range(4, 9))                    # Δt = 2^-4 … 2^-8
weak_dts = np.array([2.0**-p for p in weak_ps])
weak_mean, weak_exact_mean = [], []
for p in weak_ps:
    dt, n = 2.0**-p, 2**p
    _, Xe = euler_maruyama(a2, b2, X0, T, n, rng, n_paths=N_WEAK)    # 새 난수
    weak_mean.append(mc_estimate(Xe[:, -1]))
    weak_exact_mean.append((1 + mu2 * dt) ** n)
    if p == 4:                                  # Δt = 1/16: 2차 모멘트와 밀스타인 평균도 기록
        Xe_w16 = Xe
        _, Xm_w16 = milstein(a2, b2, bx2, X0, T, n, rng, n_paths=N_WEAK)
mom2_w16 = mc_estimate(Xe_w16[:, -1] ** 2)
mil_mean_w16 = mc_estimate(Xm_w16[:, -1])

print(f"E[X_1] exact = e^{mu2:.0f} = {np.exp(mu2 * T):.6f}")
for p, m, ex in zip(weak_ps, weak_mean, weak_exact_mean):
    print(f"Δt = 2^-{p}: E[X^EM] MC = {m}   exact (1+μΔt)^n = {ex:.6f}   weak error |MC − e^2| = {abs(m.mean - np.exp(mu2 * T)):.4f}")
print(f"Δt = 1/16: E[(X^EM)^2] MC = {mom2_w16}   |   Milstein mean = {mil_mean_w16} (= EM mean {weak_exact_mean[0]:.6f}?)")

In [ ]:
weak_err_mc = np.array([abs(m.mean - np.exp(mu2 * T)) for m in weak_mean])
weak_err_se = np.array([m.se for m in weak_mean])
weak_err_exact = np.abs(np.exp(mu2 * T) - np.array(weak_exact_mean))
slope_weak_mc, slope_weak_exact = loglog_slope(weak_dts, weak_err_mc), loglog_slope(weak_dts, weak_err_exact)
lead = np.exp(mu2 * T) * mu2**2 * T / 2 * weak_dts      # 정리 3 의 선행항 e^{μT} μ²T/2 Δt

fig, ax = plt.subplots()
ax.errorbar(weak_dts, weak_err_mc, yerr=2 * weak_err_se, fmt="o", capsize=3, label=f"|E[X_1] − mean(X^EM_1)|, MC ± 2 SE, slope {slope_weak_mc:.2f}")
ax.plot(weak_dts, weak_err_exact, "-", color="C0", alpha=0.6, label=f"exact |e² − (1+2Δt)^(1/Δt)|, slope {slope_weak_exact:.2f}")
ax.plot(weak_dts, lead, "--", color="k", label="reference slope 1: e² · 2Δt")
ax.set(xscale="log", yscale="log", xlabel="Δt", ylabel="weak error at T=1",
       title="Weak error |E[X_1] − mean(X^EM_1)| vs Δt (log-log), μ=2, σ=0.1")
ax.legend(fontsize=8)
plt.show()

약오차는 $\Delta t$ 에 비례해 줄어든다 (기울기 ≈ 1; 정확 곡선은 0.96 — 역시 유한-$\Delta t$ 효과). 가장 작은 $\Delta t$ 에서는 MC 잡음(±2 SE ≈ 0.006–0.02)이 약오차 0.057 의 몇 분의 일이라 점이 흔들린다 — $\sigma$ 를 더 키우면 약오차가 잡음에 묻힌다 (2.6 절 함정).

### Step 4 — 가정 깨기 미리보기: $dX=-X^3dt+dB$

드리프트 $-x^3$ 은 전역 Lipschitz 가 아니다 (초선형). 참해는 잘 정의되고 정상분포 $\propto e^{-x^4/2}$ 를 가지지만, EM 은 굵은 $\Delta t$ 에서 유한 확률로 폭발한다: $X_{k+1}\approx X_k(1-X_k^2\Delta t)$ 라 $X_k^2>2/\Delta t$ 이면 진폭이 매 스텝 커진다 ($\Delta t=0.5$: $|X_k|>2$ 가 문턱). 오버플로 경고를 막기 위해 `np.errstate` 안에서 돌린다.

In [ ]:
a3 = lambda t, x: -x**3
b3 = lambda t, x: 1.0 + 0 * x
T3 = 10.0
paths3 = {}
with np.errstate(all="ignore"):
    for dt in [0.5, 1 / 64]:
        t3, X3 = euler_maruyama(a3, b3, 0.0, T3, round(T3 / dt), rng, n_paths=N_PATHS)
        paths3[dt] = (t3, X3)
        frac = mc_proportion(~np.isfinite(X3[:, -1]) | (np.abs(X3[:, -1]) > 1e3))
        print(f"Δt = {dt:<8}: exploded fraction at T={T3:.0f} = {frac}   (explosion_fraction = {explosion_fraction(X3[:, -1]):.4f})")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, dt in zip(axes, [1 / 64, 0.5]):
    t3, X3 = paths3[dt]
    with np.errstate(all="ignore"):
        X_plot = np.clip(np.nan_to_num(X3, nan=1e6, posinf=1e6, neginf=-1e6), -1e6, 1e6)   # 폭발 경로는 잘라서 표시
        plot_paths(t3, X_plot, ax=ax, n_show=8, lw=1)
    ax.set(xlabel="t", ylabel="X_t", title=f"EM for dX = −X³ dt + dB, Δt = {'1/64' if dt < 0.1 else '1/2'}")
axes[1].set_yscale("symlog")
plt.show()

왼쪽($\Delta t=1/64$)은 안정하게 $[-2,2]$ 근처를 오간다; 오른쪽($\Delta t=1/2$)은 일부 경로가 문턱 $|X|>2$ 를 넘는 순간 symlog 축 밖으로 튀어 나간다 (약 19–20%). 안정한 격자에서는 EM 의 시간 평균이 정상분포의 $E[X^2]=\sqrt2\,\Gamma(3/4)/\Gamma(1/4)$ 근처에 온다 — 편향 $O(\Delta t)$ 가 있어 assert 는 하지 않고 눈으로만 본다 (E3).

In [ ]:
T_long, dt_long = 20.0, 1 / 64
with np.errstate(all="ignore"):
    t4, X4 = euler_maruyama(a3, b3, 0.0, T_long, round(T_long / dt_long), rng, n_paths=scale(2000))
burn = round(2.0 / dt_long)                      # 처음 2 시간 단위는 버림 (정상상태 도달)
x2_time_avg = float((X4[:, burn:] ** 2).mean())
x2_stationary = np.sqrt(2) * gamma(0.75) / gamma(0.25)
print(f"time average of X^2 (EM, Δt=1/64, t∈[2,{T_long:.0f}]) = {x2_time_avg:.4f}   vs stationary E[X^2] = {x2_stationary:.4f}")

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드로 계산한다: 정리 1 (`mse_em_exact`, `mse_mil_exact`), 정리 3 ($(1+\mu\Delta t)^n$, 2차 모멘트), $E[X_T]=e^{\mu T}$. 예측 항목 `em_mean_dt16` 을 위해 Step 2 의 경로를 factor 64 로 coarsen 한 EM ($\Delta t=1/16$, $\mu=1$) 을 하나 더 돌린다. RMS 행의 SE 는 델타법이다. 표는 소수 4자리로 찍히므로 MSE 행은 $10^{-4}$ 단위로 표시한다 (판정은 단위와 무관).

In [ ]:
_, Xe16 = euler_maruyama(a, b, X0, T, 16, rng, dW=np.diff(coarsen(t, B, 64)[1], axis=1))
exact = {
    "exact_mean": np.exp(mu * T),
    "em_mean_dt16": X0 * (1 + mu / 16) ** 16,
    "rmse_em_dt32": np.sqrt(mse_em_exact(mu, sigma, 1 / 32)),
    "rmse_mil_dt32": np.sqrt(mse_mil_exact(mu, sigma, 1 / 32)),
    "mom2_w16": (1 + (2 * mu2 + sigma2**2) / 16 + mu2**2 / 256) ** 16,
    "mil_mean_w16": (1 + mu2 / 16) ** 16,
    "weak_err_dt256": abs(np.exp(mu2 * T) - (1 + mu2 / 256) ** 256),
}
est = {
    "exact_mean": mc_estimate(X_T),
    "em_mean_dt16": mc_estimate(Xe16[:, -1]),
    "rmse_em_dt32": (rms_em[-1], se_rms_em[-1]),
    "rmse_mil_dt32": (rms_mil[-1], se_rms_mil[-1]),
    "mom2_w16": mom2_w16,
    "mil_mean_w16": mil_mean_w16,
    "weak_err_dt256": (weak_err_mc[-1], weak_err_se[-1]),
}

In [ ]:
x1e4 = lambda m: MCResult(m.mean * 1e4, m.se * 1e4, m.n)      # 표가 소수 4자리라 MSE 는 1e-4 단위로 표시
rows = []
for i, dt in enumerate(dts):
    rows.append({"name": f"EM MSE ×1e4, Δt=2^-{10 - i}", "predicted": None, "estimate": x1e4(mse_em[i]), "exact": 1e4 * mse_em_exact(mu, sigma, dt)})
for i, dt in enumerate(dts):
    rows.append({"name": f"Milstein MSE ×1e4, Δt=2^-{10 - i}", "predicted": None, "estimate": x1e4(mse_mil[i]), "exact": 1e4 * mse_mil_exact(mu, sigma, dt)})
for key, label in [("rmse_em_dt32", "EM RMS error, Δt=1/32"), ("rmse_mil_dt32", "Milstein RMS error, Δt=1/32")]:
    rows.append({"name": label, "predicted": predictions.get(key), "estimate": est[key][0], "se": est[key][1], "exact": exact[key]})
rows.append({"name": f"slope log RMS_EM vs log Δt (asymptotic 0.5; exact curve {slope_em_exact:.3f})", "predicted": predictions.get("slope_em"), "estimate": slope_em_rms, "exact": None})
rows.append({"name": f"slope log RMS_Mil vs log Δt (asymptotic 1.0; exact curve {slope_mil_exact:.3f})", "predicted": predictions.get("slope_mil"), "estimate": slope_mil_rms, "exact": None})
rows.append({"name": "E[X_1], μ=1, σ=0.5 (exact solution samples)", "predicted": predictions.get("exact_mean"), "estimate": est["exact_mean"], "exact": exact["exact_mean"]})
rows.append({"name": "E[X^EM_16], μ=1, σ=0.5, Δt=1/16", "predicted": predictions.get("em_mean_dt16"), "estimate": est["em_mean_dt16"], "exact": exact["em_mean_dt16"]})
for p, m, ex in zip(weak_ps, weak_mean, weak_exact_mean):
    rows.append({"name": f"E[X^EM_1], μ=2, σ=0.1, Δt=2^-{p}", "predicted": None, "estimate": m, "exact": ex})
rows.append({"name": "E[(X^EM_1)^2], μ=2, σ=0.1, Δt=1/16", "predicted": None, "estimate": est["mom2_w16"], "exact": exact["mom2_w16"]})
rows.append({"name": "E[X^Mil_1], μ=2, σ=0.1, Δt=1/16 (= EM mean)", "predicted": None, "estimate": est["mil_mean_w16"], "exact": exact["mil_mean_w16"]})
rows.append({"name": "weak error |E[X_1] − E[X^EM_256]|, Δt=2^-8", "predicted": predictions.get("weak_err_dt256"), "estimate": est["weak_err_dt256"][0], "se": est["weak_err_dt256"][1], "exact": exact["weak_err_dt256"]})
Markdown(compare_table(rows))

각 행에서 불일치가 뜻하는 것:
- **EM MSE / Milstein MSE (12 행)**: 정리 1 의 닫힌 형식이 틀렸거나, coarsen 한 증분이 같은 경로가 아니거나(그러면 오차가 $O(1)$), 스킴 구현(특히 밀스타인의 $\frac12bb_x(\Delta B^2-\Delta t)$)이 잘못된 것.
- **RMS error Δt=1/32**: 위 MSE 행의 제곱근 — 예측과의 비교용. SE 는 델타법이므로 MSE 행과 같은 판정이 나와야 한다.
- **slope 행**: 유한-$\Delta t$ 값이라 닫힌 형식 칸은 비워 두었다 (판정 "—"). 아래 셀에서 범위 $[0.4,0.7]$, $[0.85,1.15]$ 로만 검사한다. EM 이 1 근처면 밀스타인 보정을 EM 에 넣은 것, 밀스타인이 0.5 근처면 보정항이 빠진 것.
- **E[X_1] (exact samples)**: $e^{\mu T}$ 와 다르면 정확해 공식(특히 $-\sigma^2/2$)이나 `brownian_paths` 의 증분 분산 문제.
- **E[X^EM_16] (μ=1)** 과 **E[X^EM_1] (μ=2, 5 행)**: $(1+\mu\Delta t)^n$ 과 다르면 정리 3 의 독립성 논증이 코드에서 깨진 것 — 예컨대 $\Delta B_k$ 를 $X_k$ 계산 뒤가 아니라 앞에서 쓰는 비적합 스킴.
- **E[(X^EM_1)^2]**: 2차 모멘트 공식 $(1+(2\mu+\sigma^2)\Delta t+\mu^2\Delta t^2)^n$ 검증; 연속 값 $e^{(2\mu+\sigma^2)T}=e^{4.01}\approx55.1$ 과는 다른 것이 정상이다 (약오차).
- **E[X^Mil_1]**: EM 평균과 다르면 밀스타인 보정항의 평균이 0 이 아니라는 뜻 — $\Delta B^2-\Delta t$ 대신 $\Delta B^2$ 을 쓴 흔한 버그.
- **weak error Δt=2^-8**: 정확값 0.057 과 SE 안에서 맞아야 한다; 훨씬 크면 MC 잡음이 약오차를 덮는 상황 (σ 가 큰 경우).

In [ ]:
for i, dt in enumerate(dts):
    assert_close(mse_em[i], mse_em_exact(mu, sigma, dt), k=4, name=f"EM MSE dt={dt}")
    assert_close(mse_mil[i], mse_mil_exact(mu, sigma, dt), k=4, name=f"Milstein MSE dt={dt}")
for p, m, ex, dt in zip(weak_ps, weak_mean, weak_exact_mean, weak_dts):
    assert_close(m, ex, k=4, name=f"EM mean dt={dt}")
assert_close(est["mom2_w16"], exact["mom2_w16"], k=4, name="EM second moment dt=1/16")
assert_close(est["mil_mean_w16"], exact["mil_mean_w16"], k=4, name="Milstein mean dt=1/16 (= EM mean)")
assert_close(est["exact_mean"], exact["exact_mean"], k=4, name="E[X_T] exact samples")
assert_close(est["em_mean_dt16"], exact["em_mean_dt16"], k=4, name="EM mean dt=1/16, mu=1")
# 기울기는 정확한 유한-Δt 값이 없으므로 assert_close 가 아니라 범위 검사 (2.6 절 함정)
assert 0.4 <= slope_em_rms <= 0.7, f"EM RMS slope {slope_em_rms:.3f} outside [0.4, 0.7]"
assert 0.85 <= slope_mil_rms <= 1.15, f"Milstein RMS slope {slope_mil_rms:.3f} outside [0.85, 1.15]"
print(f"all assert_close passed (k=4); slopes EM {slope_em_rms:.3f} ∈ [0.4, 0.7], Milstein {slope_mil_rms:.3f} ∈ [0.85, 1.15]")

## 6. 연습문제

### E1 계산

(a) $E[X_n^{EM}]=X_0(1+\mu\Delta t)^n$ 에서 약오차의 선행항이 $X_0e^{\mu T}\frac{\mu^2T}{2}\Delta t$ 임을 보이고, $\mu=2,T=1,\Delta t=1/256$ 에서 정확값 $e^2-(1+2/256)^{256}$ 과 비교하라.
(b) EM MSE 닫힌 형식(정리 1)에서 $\log$ 전개로 선행항이 $\frac{\sigma^4}{2}Te^{(2\mu+\sigma^2)T}\Delta t$ 임을 보이고, $\mu=1,\sigma=0.5$, $\Delta t=2^{-10}$ 의 값을 정확값 $2.968\times10^{-4}$ 과 비교하라.
(c) Higham 원본의 $\mu=2,\sigma=1$ 에서 $E[X_T^4]=e^{(4\mu+6\sigma^2)T}$ 를 계산하고, 왜 강오차 MSE 추정의 상대 SE 가 커지는지 설명하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $n\log(1+\mu\Delta t)=n\big(\mu\Delta t-\frac{\mu^2\Delta t^2}{2}+O(\Delta t^3)\big)=\mu T-\frac{\mu^2T}{2}\Delta t+O(\Delta t^2)$ ($n\Delta t=T$). 지수를 취하면 $(1+\mu\Delta t)^n=e^{\mu T}(1-\frac{\mu^2T}{2}\Delta t+O(\Delta t^2))$, 따라서 약오차 $=X_0e^{\mu T}\frac{\mu^2T}{2}\Delta t+O(\Delta t^2)$. 숫자: $e^2\cdot\frac{4}{2}\cdot\frac1{256}=0.05773$ vs 정확 $0.05721$ — 1% 차이는 $O(\Delta t^2)$ 항.

(b) 정리 1 증명 (v) 와 같다: $\text{MSE}=e^{(2\mu+\sigma^2)T}\big[e^{c_1T\Delta t}-2e^{c_2T\Delta t}+1\big]$ 에 $e^{x}=1+x+\dots$ 를 넣으면 $e^{(2\mu+\sigma^2)T}(c_1-2c_2)T\Delta t$, $c_1-2c_2=\frac{\sigma^4}{2}$. $\mu=1,\sigma=0.5,T=1$: 계수 $=\frac{0.0625}{2}e^{2.25}=0.2965$, 즉 MSE $\approx0.2965\,\Delta t$; $\Delta t=2^{-10}$ 에서 $2.90\times10^{-4}$ (정확 $2.968\times10^{-4}$, 2% 차이 = $O(\Delta t^2)$ 항).

(c) $X_T^4=\exp\big(4(\mu-\frac{\sigma^2}{2})T+4\sigma B_T\big)$ 이고 $E[e^{4\sigma B_T}]=e^{8\sigma^2T}$ 이므로 $E[X_T^4]=e^{(4\mu+6\sigma^2)T}$. $\mu=2,\sigma=1$: $e^{14}\approx1.2\times10^6$, 반면 $\mu=1,\sigma=0.5$: $e^{5.5}\approx245$. 오차 $Y=X_n-X_T$ 의 제곱평균 추정의 SE 는 $\sqrt{\mathrm{Var}(Y^2)/N}\le\sqrt{E[Y^4]/N}$ 이고 $E[Y^4]$ 는 $E[X_T^4]$ 규모라, 상대 SE $\sim\sqrt{E[Y^4]}/E[Y^2]/\sqrt N$ 가 로그정규 꼬리 때문에 수십 % 로 커진다 ($e^{14}$ 는 $e^{5.5}$ 의 5000 배). 4 SE 판정이 무의미해지므로 파라미터를 바꿨다.

```python
print("(a)", np.exp(2) * 2**2 / 2 / 256, "vs", np.exp(2) - (1 + 2 / 256) ** 256)
coef = 0.5**4 / 2 * np.exp(2 * 1 + 0.5**2)
print("(b) coef", coef, "leading", coef * 2**-10, "vs exact", mse_em_exact(1.0, 0.5, 2**-10))
print("(c) E[X_T^4]: Higham", np.exp(4 * 2 + 6 * 1), " ours", np.exp(4 * 1 + 6 * 0.25))
```

</details>

### E2 유도 후 시뮬레이션 검증

가법 잡음 $dX=-X\,dt+dB$, $X_0=0$ (OU 과정, 07d 의 주인공).
(a) 밀스타인 보정항이 0 임을 보여라 — 즉 EM 과 밀스타인이 같은 스킴이다.
(b) 정확해 $X_T=\int_0^Te^{-(T-s)}dB_s\sim N\big(0,(1-e^{-2T})/2\big)$ 를 유도하라.
(c) 세밀 경로($\Delta t=2^{-12}$)의 EM 을 **참조해**로 두고, coarsen 으로 $\Delta t=2^{-8},\dots,2^{-4}$ 의 RMS 오차를 재어 강차수 ≈ 1 (정리 2, 가법 잡음) 을 확인하라. 참조해 자체의 오차가 기울기를 위로 편향시키는 이유를 설명하라. (정확해 $X_T$ 는 $B_T$ 만의 함수가 아니라서 본문처럼 한 번에 계산할 수 없다 — 그래서 참조해가 필요하다.)

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $b\equiv1$ 이므로 $b_x=0$, 보정항 $\frac12bb_x(\Delta B^2-\Delta t)=0$. 가법 잡음에서는 EM = 밀스타인이고, 정리 2 에 따라 EM 의 강차수가 이미 1 이다.

(b) $d(e^tX_t)=e^tX_t\,dt+e^t\,dX_t=e^t\,dB_t$ (곱셈 규칙; $e^t$ 는 유계변동이라 $dt\,dB=0$). 적분하면 $e^TX_T=\int_0^Te^s\,dB_s$, 즉 $X_T=\int_0^Te^{-(T-s)}dB_s$. 결정론적 피적분함수의 Itô 적분은 가우스이고 평균 0, 등거리성으로 $\mathrm{Var}=\int_0^Te^{-2(T-s)}ds=(1-e^{-2T})/2$. $T=1$: $0.43233$.

(c) 기울기는 1 을 약간 넘는다 (약 1.0–1.13). 참조해(EM, $\Delta t=2^{-12}$) 자체가 $O(2^{-12})$ 의 오차를 가지므로 가장 세밀한 수준($2^{-8}$)의 측정 오차는 "참오차 − 참조오차" 의 상관된 조합이 되어 작게 나오고, 굵은 수준은 영향이 없으므로 log-log 직선이 왼쪽 끝에서 아래로 처져 기울기가 과대평가된다. 이것이 본문에서 정확해를 쓴 이유다.

```python
a_ou = lambda t, x: -x
b_ou = lambda t, x: 1.0 + 0 * x
n_ref = 4096                                                 # Δt = 2^-12
t_ou, B_ou = brownian_paths(N_PATHS, n_ref, T, rng)
_, X_ref = euler_maruyama(a_ou, b_ou, 0.0, T, n_ref, rng, dW=np.diff(B_ou, axis=1))
print("Var(X_1): EM reference", mc_estimate(X_ref[:, -1] ** 2), " vs exact", (1 - np.exp(-2 * T)) / 2)
ou_dts, ou_rms = [], []
for factor in [16, 32, 64, 128, 256]:                        # Δt = 2^-8 … 2^-4
    tk, Bk = coarsen(t_ou, B_ou, factor)
    _, Xk = euler_maruyama(a_ou, b_ou, 0.0, T, n_ref // factor, rng, dW=np.diff(Bk, axis=1))
    ou_dts.append(factor / n_ref)
    ou_rms.append(np.sqrt(np.mean((Xk[:, -1] - X_ref[:, -1]) ** 2)))
print("RMS vs reference:", np.round(ou_rms, 5), " slope", round(loglog_slope(ou_dts, ou_rms), 3), "(expected ≈ 1, biased upward)")
```

참조해로 잰 분산 $0.43$ 은 정확값 $0.43233$ 과 SE 안에서 맞고 (EM 분산의 정확값은 $(1-(1-\Delta t)^{2n})/(2-\Delta t)$ 로 편향이 $10^{-4}$ 수준), 기울기는 1.0 근처에 온다.

</details>

### E3 가정을 깨보기

$dX=-X^3dt+dB$ 는 드리프트가 전역 Lipschitz 가 아니다 (초선형 성장). $X_0=0$, EM $\Delta t=0.5$, $T=10$ 에서 폭발($|X|>10^3$ 또는 non-finite)하는 경로 비율을 재고 $\Delta t=0.25,\ 1/64$ 와 비교하라. 왜 $|X_k|>\sqrt{2/\Delta t}$ 이면 발산하는지 보여라. 안정한 $\Delta t=1/64$ 에서 긴 경로의 시간 평균 $\overline{X^2}$ 를 정상분포 $\propto e^{-x^4/2}$ 의 $E[X^2]=\sqrt2\,\Gamma(3/4)/\Gamma(1/4)$ 과 비교하라 (Step 4 의 값을 써도 된다).

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**발산 문턱.** 잡음을 잠시 무시하면 $X_{k+1}=X_k-X_k^3\Delta t=X_k(1-X_k^2\Delta t)$. $|1-X_k^2\Delta t|>1\iff X_k^2\Delta t>2\iff|X_k|>\sqrt{2/\Delta t}$ 이면 $|X_{k+1}|>|X_k|$ 이고, 그 뒤로는 $|X|$ 가 매 스텝 세제곱으로 커져 몇 스텝 안에 오버플로한다 (이중 초과 후 되돌아올 수 없다: 곱 인수 $|1-X^2\Delta t|$ 가 점점 커진다). $\Delta t=0.5$: 문턱 $|X|>2$; 정상상태의 $X$ 는 $\pm1$ 근처를 오가고 한 스텝 잡음이 $N(0,0.5)$ 라 20 스텝 중 한 번이라도 문턱을 넘길 확률이 약 19–20%. $\Delta t=0.25$: 문턱 $\sqrt8\approx2.83$, 잡음 sd 0.5 → 40 스텝에서 약 0.04%. $\Delta t=1/64$: 문턱 $\sqrt{128}\approx11.3$ 은 사실상 도달 불가 → 0. 참해(연속)는 $-x^3$ 이 강하게 되돌려서 절대 폭발하지 않는다 — 폭발은 순전히 이산화의 산물이다 (Hutzenthaler–Jentzen–Kloeden 2011: EM 의 모멘트가 발산; 해결책은 암시적 EM 또는 tamed EM, Season 2).

**정상분포.** 생성원 $\mathcal L=-x^3\partial_x+\frac12\partial_x^2$ 의 정상밀도는 $p\propto e^{-2U(x)}$, $U=x^4/4$ (드리프트 $=-U'$, 확산 1): $p\propto e^{-x^4/2}$. $E[X^2]=\int x^2e^{-x^4/2}dx\big/\int e^{-x^4/2}dx$; 치환 $u=x^4/2$ 로 $\int_0^\infty x^{m}e^{-x^4/2}dx=2^{(m-3)/4}\frac14\Gamma(\frac{m+1}{4})$ 이므로 비는 $2^{1/2}\Gamma(3/4)/\Gamma(1/4)=0.4780$. EM $\Delta t=1/64$ 의 시간 평균은 0.47–0.48 로 정확값과 1% 안팎에서 맞는다 — 차이는 $O(\Delta t)$ 편향과 시간 평균의 통계 오차(경로 간 상관이 있어 SE 를 단순히 쓸 수 없다)의 합이다.

```python
with np.errstate(all="ignore"):
    for dt in [0.5, 0.25, 1 / 64]:
        _, X3 = euler_maruyama(a3, b3, 0.0, T3, round(T3 / dt), rng, n_paths=N_PATHS)
        print(f"Δt = {dt:<8}: threshold sqrt(2/Δt) = {np.sqrt(2 / dt):5.2f}, exploded fraction = "
              f"{mc_proportion(~np.isfinite(X3[:, -1]) | (np.abs(X3[:, -1]) > 1e3))}")
print(f"time average of X^2 (Step 4) = {x2_time_avg:.4f}  vs  sqrt(2) Γ(3/4)/Γ(1/4) = {x2_stationary:.4f}")
```

</details>

## 7. 정리

1. **EM**: $X_{k+1}=X_k+a\Delta t+b\Delta B_k$; **밀스타인**은 $+\frac12bb_x(\Delta B_k^2-\Delta t)$ — 07a 의 $\int B\,dB=\frac12(B^2-t)$ 가 그대로 보정항이다.
2. **강수렴**은 같은 경로의 오차, **약수렴**은 분포의 오차. 강오차 측정은 세밀 경로를 `coarsen` 해서 모든 해상도가 같은 $B$ 를 쓰게 해야 한다 (새 난수를 뽑으면 오차가 $O(1)$).
3. GBM 에서는 EM·밀스타인의 MSE 가 닫힌 형식으로 나오고 (정리 1, 완전 증명), 시뮬레이션이 6개 수준 모두 4 SE 이내로 맞았다: EM RMS $\asymp\Delta t^{1/2}$ (선행항 $\frac{\sigma^4}{2}Te^{(2\mu+\sigma^2)T}\Delta t$), 밀스타인 $\asymp\Delta t$ ($O(\Delta t)$ 항 소거).
4. EM 의 평균은 $(1+\mu\Delta t)^n$ (결정론적 오일러) 이라 약차수 1; 밀스타인은 평균을 바꾸지 않으므로 기댓값 계산에는 이득이 없다.
5. Lipschitz 가정이 깨지면 ($-X^3$) 굵은 $\Delta t$ 에서 EM 이 유한 확률로 폭발한다 (문턱 $|X|>\sqrt{2/\Delta t}$) — 수렴 정리의 가정은 장식이 아니다.

**Trap 카드.** Q: 밀스타인이 EM 보다 "정확"하니 옵션 가격 MC 도 밀스타인으로 해야 하나?
→ A: 아니다. 밀스타인은 강차수(경로별)만 1 로 올리고 약차수(기댓값)는 EM 과 같은 1 이다 — GBM 에서 두 스킴의 평균은 정확히 같은 $(1+\mu\Delta t)^n$ (5절 표의 `E[X^Mil_1]` 행). 기댓값이 목적이면 스텝을 줄이거나 경로 수를 늘리는 편이 낫다.

**다음 노트북: 07d (OU 과정).** 닫힌 해가 있는 선형 SDE 의 대표 예 OU 과정을 보고, 정확 전이분포로 샘플링하는 것(`ou_exact`)과 EM 의 차이 — 정상분산 $\sigma^2/(2\theta)$ vs $\sigma^2/(2\theta-\theta^2\Delta t)$ — 를 오늘 배운 약오차의 관점으로 정량화한다 (E2 의 가법 잡음 SDE 가 바로 그 과정이다).

`log/progress.md` 한 줄 템플릿:
`- [ ] 07c 오일러-마루야마·밀스타인 — 날짜: ____ / 소요: ___분 / 예측 적중: _/7 / 메모: ____`